In [1]:
import json
import uuid
from sqlalchemy import create_engine

from utils import reset_db, get_session, model_to_dict
from data.models import udahub

# Udahub Application

## Core Database

**Init DB**

In [2]:
udahub_db = "data/core/udahub.db"

In [3]:
reset_db(udahub_db)

2026-11-07 13:37:45,231 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-11-07 13:37:45,232 INFO sqlalchemy.engine.Engine COMMIT
✅ Recreated data/core/udahub.db with fresh schema


In [4]:
engine = create_engine(f"sqlite:///{udahub_db}", echo=False)
udahub.Base.metadata.create_all(bind=engine)

**Account**

In [5]:
account_id = "cultpass"
account_name = "CultPass Card"

In [6]:
with get_session(engine) as session:
    account = udahub.Account(
        account_id=account_id,
        account_name=account_name,
    )
    session.add(account)

## Integrations

**Knowledge Base**

In [13]:
# TODO: Create additional 10 articles
import json
from pathlib import Path

articles_path = Path("data/external/cultpass_articles.jsonl")

new_articles = [
    {"title": "Refund and Billing Dispute Policy",
     "content": "Refunds are NOT issued by the support bot.\n\n- Monthly fees are non-refundable once the billing cycle has started\n- Exceptions (duplicate charge, charge after confirmed cancellation) must be approved by the support lead\n- Never promise a refund; collect the charge date and amount and escalate to human support\n\n**Suggested phrasing:**\n\"I understand your concern about this charge. Refund requests are reviewed by our support team, so I'll pass your details to them and they'll follow up.\"",
     "tags": "billing, refund, charge, dispute, escalation"},
    {"title": "Accepted Payment Methods and Failed Payments",
     "content": "CultPass accepts major credit/debit cards and Pix (Brazil).\n\n- Update the payment method in the app > 'My Account' > 'Payment'\n- A failed payment is retried automatically after 3 days; the subscription stays active for a 5-day grace period\n- Common causes: expired card, insufficient funds, bank blocking international charges\n- If the card is valid and the payment keeps failing, escalate to human support\n\n**Suggested phrasing:**\n\"Your payment may have failed because of an expired card or a bank restriction. You can update your payment method under 'My Account' > 'Payment', and we'll retry automatically.\"",
     "tags": "billing, payment, card, pix, failed payment"},
    {"title": "Basic vs Premium Tiers and Premium Experiences",
     "content": "CultPass has two tiers: Basic and Premium.\n\n- Basic: access to standard experiences\n- Premium: access to standard AND premium experiences (e.g. exclusive tours, limited-capacity events)\n- Basic members who try to reserve a premium experience are asked to upgrade\n- Upgrade via app > 'My Account' > 'Manage Plan'; the new tier applies immediately and is billed pro-rata\n\n**Suggested phrasing:**\n\"Premium experiences are available on the Premium tier. You can upgrade anytime under 'My Account' > 'Manage Plan'.\"",
     "tags": "premium, tier, upgrade, subscription, experiences"},
    {"title": "QR Code Not Working at Check-in",
     "content": "If the QR code does not scan at the venue:\n\n- Increase screen brightness and remove screen protectors/glare\n- Make sure the reservation shows status 'Reserved' in the app (not 'Pending' or 'Cancelled')\n- Refresh the 'My Reservations' tab or log out and in again\n- Venue staff can look up the reservation by the customer's name and email\n- If still failing, escalate to human support\n\n**Suggested phrasing:**\n\"Please raise your screen brightness and refresh 'My Reservations'. If the code still won't scan, show your name and email to the venue staff, who can check you in manually.\"",
     "tags": "qr code, check-in, attendance, reservation, troubleshooting"},
    {"title": "App Crashes, Freezes or Won't Load",
     "content": "Technical troubleshooting steps for the CultPass app:\n\n- Update to the latest app version from the App Store / Google Play\n- Force-close and reopen the app; restart the phone\n- Check internet connection; try Wi-Fi vs mobile data\n- Clear cache (Android) or reinstall the app (reservations are stored in the cloud and will not be lost)\n- If it persists, escalate with device model, OS version and app version\n\n**Suggested phrasing:**\n\"Please update the app, restart it and check your connection. Reinstalling is safe since your reservations are stored in your account.\"",
     "tags": "technical, app, crash, bug, troubleshooting"},
    {"title": "How to Delete Your Account and Data Privacy",
     "content": "Account deletion and privacy requests:\n\n- Delete via app > 'My Account' > 'Privacy' > 'Delete Account'\n- Active subscriptions must be cancelled first; deletion is permanent after a 14-day recovery window\n- CultPass follows LGPD/GDPR: users may request a copy of their data or its removal\n- Data export requests are handled by human support (allow up to 15 days)\n\n**Suggested phrasing:**\n\"You can delete your account in 'My Account' > 'Privacy'. Please cancel any active subscription first. Deletion becomes permanent after 14 days.\"",
     "tags": "account, delete, privacy, lgpd, gdpr, data"},
    {"title": "Event Cancelled or Rescheduled by the Venue",
     "content": "When a partner venue cancels or reschedules an experience:\n\n- Customers are notified by push notification and email\n- Cancelled events return the credit to the monthly quota automatically\n- For a reschedule, the reservation moves to the new date; the customer can cancel it if the new date does not suit them\n- Do not promise compensation beyond the quota credit\n\n**Suggested phrasing:**\n\"If a venue cancels, the credit returns to your monthly quota automatically. If it is rescheduled, you can keep the new date or cancel and choose another experience.\"",
     "tags": "events, cancelled, rescheduled, quota, venue"},
    {"title": "Blocked or Suspended Accounts",
     "content": "An account may be blocked for policy violations (e.g. repeated no-shows, ticket resale, payment fraud).\n\n- Blocked users cannot reserve experiences or change their plan\n- Support agents must NOT reveal internal fraud details\n- Never unblock an account; ask the user to wait for a human review and escalate to human support\n\n**Suggested phrasing:**\n\"Your account is currently under review, so some actions are unavailable. I've asked our support team to look into it.\"",
     "tags": "blocked, suspended, account, policy, escalation"},
    {"title": "No-Show and Reservation Cancellation Policy",
     "content": "Cancelling a reservation:\n\n- Cancel from app > 'My Reservations' at least 12 hours before the event to get the credit back\n- Late cancellations and no-shows consume the monthly credit\n- Three no-shows within 60 days may lead to temporary suspension\n\n**Suggested phrasing:**\n\"You can cancel from 'My Reservations' up to 12 hours before the event to keep your credit. Later cancellations or no-shows use one credit.\"",
     "tags": "cancel reservation, no-show, quota, policy"},
    {"title": "Accessibility and Special Assistance at Experiences",
     "content": "CultPass encourages inclusive access:\n\n- Each experience page lists accessibility info (wheelchair access, audio description, sign language)\n- Users can add an accessibility note when reserving so the venue can prepare\n- For needs not listed, escalate to human support, which contacts the venue directly\n\n**Suggested phrasing:**\n\"Each experience lists its accessibility details. You can add a note when you reserve, and our team can contact the venue for specific needs.\"",
     "tags": "accessibility, inclusion, special needs, venue"},
    {"title": "Changing Your Email, Name or Profile Details",
     "content": "Profile updates:\n\n- Name and phone: app > 'My Account' > 'Profile'\n- Changing the login email requires verification via a code sent to the NEW address\n- If the user no longer has access to the old email, escalate to human support for identity verification\n\n**Suggested phrasing:**\n\"You can update your profile in 'My Account' > 'Profile'. To change your login email we'll send a verification code to the new address.\"",
     "tags": "account, profile, email, update, verification"},
]

# Append only the articles that are not already in the file (safe to re-run)
existing_titles = set()
if articles_path.exists():
    with open(articles_path, "r", encoding="utf-8") as f:
        existing_titles = {json.loads(line)["title"] for line in f if line.strip()}

added = 0
with open(articles_path, "a", encoding="utf-8") as f:
    for article in new_articles:
        if article["title"] not in existing_titles:
            f.write(json.dumps(article, ensure_ascii=False) + "\n")
            added += 1

print(f"Added {added} new articles")

Added 11 new articles


In [14]:
cultpass_articles = []

with open('data/external/cultpass_articles.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_articles.append(json.loads(line))

In [15]:
cultpass_articles

[{'title': 'How to Reserve a Spot for an Event',
  'content': 'If a user asks how to reserve an event:\n\n- Guide them to the CultPass app\n- Instruct them to browse the experience catalog and tap \'Reserve\'\n- If it\'s a premium or limited event, check if reservation confirmation is required via email\n- Remind them to arrive at least 15 minutes early with their QR code visible\n\n**Suggested phrasing:**\n"You can reserve an experience by opening the CultPass app, selecting your desired event, and tapping \'Reserve\'. Be sure to arrive 15 minutes early with your QR code ready."',
  'tags': 'reservation, events, booking, attendance'},
 {'title': "What's Included in a CultPass Subscription",
  'content': 'Each user is entitled to 4 cultural experiences per month, which may include:\n- Art exhibitions\n- Museum entries\n- Music concerts\n- Film screenings and more\n\nSome premium experiences may require an additional fee (visible in the app).\n\n**Suggested phrasing:**\n"Your CultPass s

In [16]:
if len(cultpass_articles) < 14:
    raise AssertionError("You should load the articles with at least 14 records")

In [17]:
with get_session(engine) as session:
    kb = []
    for article in cultpass_articles:
        knowledge = udahub.Knowledge(
            article_id=str(uuid.uuid4()),
            account_id=account_id,
            title=article["title"],
            content=article["content"],
            tags=article["tags"]
        )
        kb.append(knowledge)
    session.add_all(kb) 
    

**Ticket**

In [18]:
cultpass_users = []

with open('data/external/cultpass_users.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_users.append(json.loads(line))

In [19]:
ticket_info = {
    "status": "open",
    "content": "I can't log in to my Cultpass account.",
    "owner_id": cultpass_users[0]["id"],
    "owner_name": cultpass_users[0]["name"],
    "role": "user",
    "channel": "chat",
    "tags": "login, access",
}

In [20]:
with get_session(engine) as session:
    user = session.query(udahub.User).filter_by(
        account_id=account_id,
        external_user_id=ticket_info["owner_id"],
    ).first()

    if not user:
        user = udahub.User(
            user_id=str(uuid.uuid4()),
            account_id=account_id,
            external_user_id=ticket_info["owner_id"],
            user_name=ticket_info["owner_name"],
        )
    
    ticket = udahub.Ticket(
        ticket_id=str(uuid.uuid4()),
        account_id=account_id,
        user_id=user.user_id,
        channel=ticket_info["channel"],
    )
    metadata = udahub.TicketMetadata(
        ticket_id=ticket.ticket_id,
        status=ticket_info["status"],
        main_issue_type=None,
        tags=ticket_info["tags"],
    )

    first_message = udahub.TicketMessage(
        message_id=str(uuid.uuid4()),
        ticket_id=ticket.ticket_id,
        role=ticket_info["role"],
        content=ticket_info["content"],
    )

    session.add_all([user, ticket, metadata, first_message])


# Tests

In [21]:
with get_session(engine) as session:
    account = session.query(udahub.Account).filter_by(
        account_id=account_id
    ).first()
    print(account)

<Account(account_id='cultpass', account_name='CultPass Card')>


In [22]:
with get_session(engine) as session:
    account = session.query(udahub.Account).filter_by(
        account_id=account_id
    ).first()
    for article in account.knowledge_articles:
        print(article)

<Knowledge(article_id='ccc290c1-7a1f-4b4b-8c7a-7cabd0ca3402', title='How to Reserve a Spot for an Event')>
<Knowledge(article_id='d26d7661-30e4-452d-87e1-9d05ca483d11', title='What's Included in a CultPass Subscription')>
<Knowledge(article_id='5bd6a117-2e07-4d3d-86e0-38fb44147054', title='How to Cancel or Pause a Subscription')>
<Knowledge(article_id='7d63de11-04cd-4383-9ca8-9e88c0718703', title='How to Handle Login Issues?')>
<Knowledge(article_id='61eefa62-7edc-40ce-a04a-ca6f2b83b44e', title='Refund and Billing Dispute Policy')>
<Knowledge(article_id='2a9e2bb8-da5b-4d98-a603-e4da74f20467', title='Accepted Payment Methods and Failed Payments')>
<Knowledge(article_id='7234e6dc-3da5-480c-a628-ab22a4126020', title='Basic vs Premium Tiers and Premium Experiences')>
<Knowledge(article_id='111d8c8e-7660-4014-bea0-6ae01b704a57', title='QR Code Not Working at Check-in')>
<Knowledge(article_id='285f0eae-bd28-4e7f-a3da-cadc780ebf41', title='App Crashes, Freezes or Won't Load')>
<Knowledge(arti

In [23]:
with get_session(engine) as session:
    users = session.query(udahub.User).all()
    for user in users:
        print(user)

<User(user_id='65e0f90d-a311-49d7-848f-baa0f8eb2a11', user_name='Alice Kingsley', external_user_id='a4ab87')>


In [24]:
with get_session(engine) as session:
    user = session.query(udahub.User).filter_by(
        account_id=account_id,
        external_user_id=ticket_info["owner_id"],
    ).first()
    
    ticket:udahub.Ticket = user.tickets[0]
    for message in ticket.messages:
        print(message)

<TicketMessage(message_id='e65df215-a983-4e3a-bea6-4e2cdb9f79e1', role='user', content='I can't log in to my Cultpass ...')>
